In [2]:
import json
import pandas as pd
from pathlib import Path

DEBUG_DIR = Path("/home/hello/Projects/Statements/code/debug")
ATOM_FILTER = "X6"
OUT_CSV = DEBUG_DIR / f"debug_full_{ATOM_FILTER}.csv"

KW = [
    "protected disclosure",
    "breach of legal obligation",
    "reasonable belief",
    "whistleblowing",
    "PIDA",
]

def read_text(p: Path) -> str:
    return p.read_text(encoding="utf-8", errors="replace")

def read_json(p: Path):
    return json.loads(read_text(p))

def kw_hits(text: str):
    t = (text or "").lower()
    return {f"has_{k.replace(' ','_')}": (k.lower() in t) for k in KW}

def extract_first_n_paras(evidence_pack: dict, n=3):
    paras = (evidence_pack or {}).get("paras") or []
    out = []
    for p in paras[:n]:
        out.append({
            "para_id": p.get("para_id"),
            "text_head": (p.get("text") or "")[:200].replace("\n", " "),
        })
    return out

# ---- show what the directory contains for this atom ----
patterns = {
    "pre_prompt": f"*__{ATOM_FILTER}__iter*__pre_llm__prompt.txt",
    "pre_evidence": f"*__{ATOM_FILTER}__iter*__pre_llm__evidence_pack.json",
    "post_verdict": f"*__{ATOM_FILTER}__iter*__post_llm__verdict.json",
    "post_anchor_check": f"*__{ATOM_FILTER}__iter*__post_llm__anchor_check.json",
}
for k, pat in patterns.items():
    n = len(list(DEBUG_DIR.glob(pat)))
    print(k, "matches:", n, "| pattern:", pat)

# ---- collect by (doc_slug, atom_id, iter) ----
records = {}

def parse_key_from_filename(name: str):
    # expected: <doc_slug>__<atom_id>__iterNN__<stage>__<type>.<ext>
    parts = name.split("__")
    if len(parts) < 4:
        return None
    doc_slug = parts[0]
    atom_id = parts[1]
    iter_part = parts[2]  # "iter03"
    if not iter_part.startswith("iter"):
        return None
    try:
        iter_i = int(iter_part.replace("iter", ""))
    except Exception:
        return None
    return (doc_slug, atom_id, iter_i)

# 1) prompts
for p in DEBUG_DIR.glob(patterns["pre_prompt"]):
    key = parse_key_from_filename(p.name)
    if not key:
        continue
    rec = records.setdefault(key, {})
    prompt_text = read_text(p)
    rec["prompt_path"] = str(p)
    rec["prompt_chars"] = len(prompt_text)
    rec.update(kw_hits(prompt_text))

# 2) evidence packs
for p in DEBUG_DIR.glob(patterns["pre_evidence"]):
    key = parse_key_from_filename(p.name)
    if not key:
        continue
    rec = records.setdefault(key, {})
    ep = read_json(p)
    rec["evidence_pack_path"] = str(p)
    rec["ep_doc_id"] = ep.get("doc_id")
    rec["ep_mode"] = ep.get("mode")

    r = ep.get("retrieval") or {}
    rec["ep_method"] = r.get("method")
    rec["ep_score"] = r.get("score")
    rec["ep_matched_paras_count"] = r.get("matched_paras")
    rec["ep_window_size"] = r.get("window_size")
    rec["ep_stride"] = r.get("stride")
    rec["ep_top_windows"] = r.get("top_windows")
    rec["ep_picked_windows"] = r.get("picked_windows")

    paras = ep.get("paras") or []
    rec["ep_n_paras"] = len(paras)

    evidence_text = "\n".join([(x.get("text") or "") for x in paras])
    rec["evidence_chars"] = len(evidence_text)
    rec.update({k.replace("has_", "evidence_has_"): v for k, v in kw_hits(evidence_text).items()})
    rec["evidence_preview_first3"] = json.dumps(extract_first_n_paras(ep, n=3), ensure_ascii=False)

# 3) post_llm verdict
for p in DEBUG_DIR.glob(patterns["post_verdict"]):
    key = parse_key_from_filename(p.name)
    if not key:
        continue
    rec = records.setdefault(key, {})
    v = read_json(p)
    rec["post_verdict_path"] = str(p)

    meta = v.get("meta") or {}
    basis = v.get("x_classification_basis") or {}
    verdict = v.get("verdict") or {}

    rec["doc_id"] = meta.get("doc_id") or rec.get("ep_doc_id")
    rec["atom_id"] = meta.get("atom_id") or key[1]
    rec["iter"] = meta.get("iter") or key[2]
    rec["llm_temperature"] = meta.get("llm_temperature")

    rec["basis_target_atom_id"] = basis.get("target_atom_id")
    rec["basis_relevant"] = basis.get("relevant")
    rec["basis_use_mode"] = basis.get("use_mode")
    rec["basis_precedent_score"] = basis.get("precedent_score")
    rec["basis_confidence"] = basis.get("confidence")
    rec["basis_matched_X"] = json.dumps(basis.get("matched_X") or [], ensure_ascii=False)
    rec["basis_note"] = basis.get("note")

    anchors = verdict.get("anchors") or []
    rec["n_anchors"] = len(anchors)
    rec["anchors"] = json.dumps(anchors, ensure_ascii=False)

    rec["verdict_retrieval_score"] = verdict.get("retrieval_score")
    rec["verdict_retrieval_method"] = verdict.get("retrieval_method")

# 4) post_llm anchor_check
for p in DEBUG_DIR.glob(patterns["post_anchor_check"]):
    key = parse_key_from_filename(p.name)
    if not key:
        continue
    rec = records.setdefault(key, {})
    a = read_json(p)
    rec["anchor_check_path"] = str(p)
    rec["anchors_ok"] = a.get("anchors_ok")
    rec["anchors_bad"] = json.dumps(a.get("anchors_bad"), ensure_ascii=False)

# ---- dataframe ----
df = pd.DataFrame(list(records.values()))
print("Built df rows:", len(df), "cols:", len(df.columns))
print("Columns:", list(df.columns))

# Only filter if present
if "post_verdict_path" in df.columns:
    df = df[df["post_verdict_path"].notna()].copy()

# Safe sort (only if present)
sort_cols = [c for c in ["doc_id", "atom_id", "iter"] if c in df.columns]
if sort_cols:
    df = df.sort_values(sort_cols)

df.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)
df.head(10)

pre_prompt matches: 3 | pattern: *__X6__iter*__pre_llm__prompt.txt
pre_evidence matches: 3 | pattern: *__X6__iter*__pre_llm__evidence_pack.json
post_verdict matches: 3 | pattern: *__X6__iter*__post_llm__verdict.json
post_anchor_check matches: 3 | pattern: *__X6__iter*__post_llm__anchor_check.json
Built df rows: 0 cols: 0
Columns: []
Saved: /home/hello/Projects/Statements/code/debug/debug_full_X6.csv


""
